# Transformer-based SMILES Generator Training

**Project**: DeepSolve - Molecule Generation for Cancer Drug Discovery  
**Goal**: Train a Transformer model to generate SMILES with solubility > 1 logS  
**Dataset**: 100k ChEMBL molecules (drug-like)

---

## Setup Instructions

1. **Upload to Google Colab**
2. **Enable GPU**: Runtime → Change runtime type → GPU (T4 or better)
3. **Run all cells** (takes ~4-8 hours for full training)
4. **Download trained model** from outputs

---

## 1. Install Dependencies & Clone Repository

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q rdkit
!pip install -q transformers
!pip install -q pandas numpy matplotlib seaborn tqdm

print("✓ Dependencies installed")

In [ ]:
# Clone repository to access dataset and utils
!git clone https://github.com/gihanpanapitiya/deepsolve-test.git
%cd deepsolve-test

print("✓ Repository cloned")

In [ ]:
# Verify GPU availability
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Please enable GPU: Runtime → Change runtime type → GPU")

## 2. Load and Prepare Dataset

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Load training and validation data
df_train = pd.read_csv('data/zinc15_100k_train.csv')
df_val = pd.read_csv('data/zinc15_100k_val.csv')

print(f"Training set: {len(df_train):,} molecules")
print(f"Validation set: {len(df_val):,} molecules")

# Sample SMILES
print("\nSample SMILES:")
for i, smi in enumerate(df_train['smiles'].head(5), 1):
    print(f"{i}. {smi}")

In [ ]:
# Analyze SMILES length distribution
train_lengths = df_train['smiles'].str.len()
val_lengths = df_val['smiles'].str.len()

print("SMILES Length Statistics:")
print(f"  Min: {train_lengths.min()}")
print(f"  Max: {train_lengths.max()}")
print(f"  Mean: {train_lengths.mean():.1f}")
print(f"  Median: {train_lengths.median():.0f}")

# Plot distribution
plt.figure(figsize=(10, 4))
plt.hist(train_lengths, bins=50, alpha=0.7, label='Train', edgecolor='black')
plt.hist(val_lengths, bins=50, alpha=0.7, label='Val', edgecolor='black')
plt.xlabel('SMILES Length')
plt.ylabel('Frequency')
plt.title('SMILES Length Distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Set maximum sequence length (cover 95% of molecules)
MAX_SEQ_LEN = int(train_lengths.quantile(0.95))
print(f"\nMax sequence length (95th percentile): {MAX_SEQ_LEN}")

## 3. Build Vocabulary (Character-level Tokenization)

In [ ]:
# Build character vocabulary from training set
all_chars = Counter()
for smi in df_train['smiles']:
    all_chars.update(smi)

# Special tokens
SPECIAL_TOKENS = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']

# Create character to index mapping
vocab = SPECIAL_TOKENS + sorted(all_chars.keys())
char_to_idx = {ch: idx for idx, ch in enumerate(vocab)}
idx_to_char = {idx: ch for ch, idx in char_to_idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX = char_to_idx['<PAD>']
SOS_IDX = char_to_idx['<SOS>']
EOS_IDX = char_to_idx['<EOS>']
UNK_IDX = char_to_idx['<UNK>']

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"\nMost common characters:")
for ch, count in all_chars.most_common(20):
    print(f"  '{ch}': {count:,}")

In [ ]:
# Tokenization functions
def smiles_to_indices(smiles, max_len=MAX_SEQ_LEN):
    """Convert SMILES string to indices."""
    indices = [SOS_IDX]  # Start token
    for ch in smiles[:max_len-2]:  # Leave room for SOS and EOS
        indices.append(char_to_idx.get(ch, UNK_IDX))
    indices.append(EOS_IDX)  # End token
    
    # Pad to max length
    while len(indices) < max_len:
        indices.append(PAD_IDX)
    
    return indices[:max_len]

def indices_to_smiles(indices):
    """Convert indices back to SMILES string."""
    chars = []
    for idx in indices:
        if idx == EOS_IDX:
            break
        if idx not in [PAD_IDX, SOS_IDX]:
            chars.append(idx_to_char.get(idx, ''))
    return ''.join(chars)

# Test tokenization
test_smiles = df_train['smiles'].iloc[0]
test_indices = smiles_to_indices(test_smiles)
reconstructed = indices_to_smiles(test_indices)

print("Tokenization test:")
print(f"Original: {test_smiles}")
print(f"Indices: {test_indices[:20]}...")
print(f"Reconstructed: {reconstructed}")
print(f"Match: {test_smiles == reconstructed}")

## 4. Create PyTorch Dataset and DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SMILESDataset(Dataset):
    def __init__(self, smiles_list, max_len=MAX_SEQ_LEN):
        self.smiles_list = smiles_list
        self.max_len = max_len
    
    def __len__(self):
        return len(self.smiles_list)
    
    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        indices = smiles_to_indices(smiles, self.max_len)
        
        # Input: all tokens except last
        # Target: all tokens except first (shifted by 1)
        input_seq = torch.tensor(indices[:-1], dtype=torch.long)
        target_seq = torch.tensor(indices[1:], dtype=torch.long)
        
        return input_seq, target_seq

# Create datasets
train_dataset = SMILESDataset(df_train['smiles'].tolist())
val_dataset = SMILESDataset(df_val['smiles'].tolist())

# Create dataloaders
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Test dataloader
sample_batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Input: {sample_batch[0].shape}")
print(f"  Target: {sample_batch[1].shape}")

## 5. Define Transformer Model (GPT-style)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class SMILESTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, nhead=8, num_layers=6, 
                 dim_feedforward=2048, dropout=0.1, max_len=MAX_SEQ_LEN):
        super().__init__()
        
        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        self.d_model = d_model
        
        # Transformer decoder layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        
        # Output layer
        self.fc_out = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
        
    def generate_square_subsequent_mask(self, sz):
        """Generate causal mask to prevent attending to future tokens."""
        mask = torch.triu(torch.ones(sz, sz), diagonal=1).bool()
        return mask
    
    def forward(self, src, tgt_mask=None):
        # Embedding + positional encoding
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        src = self.dropout(src)
        
        # Create causal mask if not provided
        if tgt_mask is None:
            tgt_mask = self.generate_square_subsequent_mask(src.size(1)).to(src.device)
        
        # Transformer decoder (self-attention only, no cross-attention)
        memory = torch.zeros_like(src)  # Dummy memory for decoder
        output = self.transformer_decoder(
            tgt=src,
            memory=memory,
            tgt_mask=tgt_mask
        )
        
        # Output projection
        logits = self.fc_out(output)
        
        return logits

# Initialize model
model = SMILESTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=512,
    nhead=8,
    num_layers=6,
    dim_feedforward=2048,
    dropout=0.1
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1e6:.1f} MB (FP32)")

## 6. Training Setup

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Loss function (ignore padding)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

# Learning rate scheduler (warmup + decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

# Training hyperparameters
NUM_EPOCHS = 20
CLIP_GRAD = 1.0

print("Training configuration:")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {optimizer.param_groups[0]['lr']}")
print(f"  Gradient clipping: {CLIP_GRAD}")

## 7. Training Loop

In [ ]:
# Training function
def train_epoch(model, loader, criterion, optimizer, device, clip_grad=1.0):
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc='Training')
    for batch_idx, (input_seq, target_seq) in enumerate(pbar):
        input_seq = input_seq.to(device)
        target_seq = target_seq.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_seq)
        
        # Calculate loss
        loss = criterion(logits.view(-1, VOCAB_SIZE), target_seq.view(-1))
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

# Validation function
def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for input_seq, target_seq in tqdm(loader, desc='Validation'):
            input_seq = input_seq.to(device)
            target_seq = target_seq.to(device)
            
            logits = model(input_seq)
            loss = criterion(logits.view(-1, VOCAB_SIZE), target_seq.view(-1))
            
            total_loss += loss.item()
    
    return total_loss / len(loader)

# SMILES validity check
def check_validity(smiles_list):
    valid = 0
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            valid += 1
    return valid / len(smiles_list) * 100

# Generate sample SMILES
def generate_samples(model, device, n_samples=100, max_len=MAX_SEQ_LEN, temperature=1.0):
    model.eval()
    generated = []
    
    with torch.no_grad():
        for _ in range(n_samples):
            # Start with <SOS> token
            seq = [SOS_IDX]
            
            for _ in range(max_len - 1):
                input_seq = torch.tensor([seq], dtype=torch.long).to(device)
                logits = model(input_seq)
                
                # Sample next token
                next_token_logits = logits[0, -1, :] / temperature
                probs = F.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, 1).item()
                
                seq.append(next_token)
                
                if next_token == EOS_IDX:
                    break
            
            smiles = indices_to_smiles(seq)
            generated.append(smiles)
    
    return generated

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'validity': [],
    'lr': []
}

best_val_loss = float('inf')

print("Starting training...")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 70)
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device, CLIP_GRAD)
    
    # Validate
    val_loss = validate_epoch(model, val_loader, criterion, device)
    
    # Generate samples and check validity
    samples = generate_samples(model, device, n_samples=100)
    validity = check_validity(samples)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Store history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['validity'].append(validity)
    history['lr'].append(current_lr)
    
    # Print metrics
    print(f"\nMetrics:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  SMILES Validity: {validity:.1f}%")
    print(f"  Learning Rate: {current_lr:.2e}")
    
    # Print sample SMILES
    print(f"\nSample Generated SMILES:")
    for i, smi in enumerate(samples[:3], 1):
        mol = Chem.MolFromSmiles(smi)
        valid_str = "✓" if mol else "✗"
        print(f"  {i}. {smi} {valid_str}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'validity': validity
        }, 'best_model.pt')
        print("  ✓ Saved best model")
    
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'history': history
        }, f'checkpoint_epoch_{epoch+1}.pt')
        print(f"  ✓ Saved checkpoint")

print("\n" + "=" * 70)
print("Training complete!")
print(f"Best validation loss: {best_val_loss:.4f}")

## 8. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train', marker='o')
axes[0].plot(history['val_loss'], label='Val', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Validity curve
axes[1].plot(history['validity'], marker='o', color='green')
axes[1].axhline(y=90, color='r', linestyle='--', label='90% target')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validity (%)')
axes[1].set_title('SMILES Validity')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Learning rate
axes[2].plot(history['lr'], marker='o', color='orange')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Final SMILES validity: {history['validity'][-1]:.1f}%")

## 9. Generate and Validate Final Samples

In [ ]:
# Load best model
checkpoint = torch.load('best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded best model from epoch {checkpoint['epoch']+1}")
print(f"  Validation loss: {checkpoint['val_loss']:.4f}")
print(f"  SMILES validity: {checkpoint['validity']:.1f}%")

# Generate large sample
print("\nGenerating 1000 SMILES...")
generated_smiles = generate_samples(model, device, n_samples=1000, temperature=0.8)

# Validate
validity = check_validity(generated_smiles)
print(f"Validity: {validity:.1f}%")

# Filter valid SMILES
valid_smiles = [smi for smi in generated_smiles if Chem.MolFromSmiles(smi) is not None]
print(f"Valid SMILES: {len(valid_smiles)} / {len(generated_smiles)}")

# Remove duplicates
unique_smiles = list(set(valid_smiles))
print(f"Unique SMILES: {len(unique_smiles)}")

# Save generated SMILES
pd.DataFrame({'smiles': unique_smiles}).to_csv('generated_smiles.csv', index=False)
print("\n✓ Saved generated SMILES to generated_smiles.csv")

## 10. Download Trained Model

Run this cell to download the trained model to your local machine:

In [ ]:
from google.colab import files

# Download best model
print("Downloading trained model...")
files.download('best_model.pt')

# Download generated SMILES
print("Downloading generated SMILES...")
files.download('generated_smiles.csv')

# Download training curves
print("Downloading training curves...")
files.download('training_curves.png')

print("\n✓ All files downloaded!")

---

## Next Steps

1. **Upload trained model** to GitHub repository
2. **Phase 3.3**: Implement RL fine-tuning with solubility constraints
3. **Phase 4**: Generate large library and analyze results

---

**Expected Results**:
- Training time: 4-8 hours on T4 GPU
- SMILES validity: >90%
- Model size: ~45 MB

**Troubleshooting**:
- If GPU runs out of memory: Reduce `BATCH_SIZE` to 64 or 32
- If validity is low (<80%): Train for more epochs or adjust temperature
- If loss plateaus: Adjust learning rate or use larger model

---